# LTN-CBM v4: Controlled LTN Regularisation for Concept Bottleneck Models
## CUB-200-2011 · Sequential, train-only rule construction

**Research question:** for an identical polarity-normalised K-literal `SpeciesModel`, does
CE + LTN implication/SatAgg improve accuracy or interventions over the same model trained
with CE only? A parameter-matched pairwise AND/OR tree is a secondary architectural baseline.

| # | Model | Concept access | Training signal |
|---|---|---|---|
| 1 | Linear CBM | all 312 | CE |
| 2 | MLP CBM | all 312 | CE |
| 3 | Oracle CBM | all 312 ground-truth attributes | CE |
| 4 | K-literal predicate | K selected, polarity-normalised | CE only (primary control) |
| 5 | Pairwise-Tree | K selected, polarity-normalised | CE |
| 6 | **LTN-CBM** | K selected, polarity-normalised | **CE + λ·SatAgg** |

The K ablation is trained and selected on validation before the single final test cell is run.


In [ ]:
# ============================
# Installation & Imports
# ============================
# This notebook requires the PyTorch implementation, LTNtorch. The unrelated
# `ltn` PyPI package is TensorFlow-based and does not provide ltn.Connective.
import sys
!{sys.executable} -m pip uninstall -q -y ltn
!{sys.executable} -m pip install -q --no-deps --force-reinstall "LTNtorch==1.0.2"
!{sys.executable} -m pip install -q scikit-learn

import os, sys, time, json, random, warnings, gc, math
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import f1_score, roc_auc_score
import ltn

assert all(hasattr(ltn, name) for name in ('Predicate', 'Connective', 'Quantifier')), (
    'LTNtorch did not import correctly; restart the kernel and rerun this cell.')
warnings.filterwarnings('ignore')
print(f'PyTorch {torch.__version__} | LTNtorch {getattr(ltn, "__version__", "installed")}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} '
          f'({torch.cuda.get_device_properties(0).total_mem/1e9:.0f} GB)')


In [ ]:
# ============================
# Configuration
# ============================
_CANDIDATES = [
    Path('/kaggle/input/datasets/faaizhussain/cub-200-2011/CUB_200_2011'),
    Path('/kaggle/input/cub2002011/CUB_200_2011/CUB_200_2011'),
    Path('/kaggle/input/caltech-birds-2011/CUB_200_2011/CUB_200_2011'),
    Path('./cub-200-2011/CUB_200_2011'),
    Path('./data/CUB_200_2011'),
]
DATA_DIR = next((p for p in _CANDIDATES if (p / 'images.txt').exists()), None)
assert DATA_DIR is not None, (
    'CUB-200-2011 not found. Add it on Kaggle or place it at '
    './cub-200-2011/CUB_200_2011 (or ./data/CUB_200_2011).')
print(f'Data: {DATA_DIR}')

OUTPUT_DIR = Path('./outputs'); OUTPUT_DIR.mkdir(exist_ok=True)
NUM_CONCEPTS, NUM_CLASSES = 312, 200
VAL_FRAC = 0.2

BATCH_SIZE     = 256
EPOCHS_CONCEPT = 30
EPOCHS_HEAD    = 30
BACKBONE_LR    = 1e-4
CONCEPT_LR     = 1e-3
HEAD_LR        = 1e-3
WEIGHT_DECAY   = 1e-4
LAMBDA_SAT     = 0.5
TOP_K          = 5
ATTRIBUTE_LABEL_POLICY = 'all'  # use all CUB labels; certainty is recorded and reported

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## 1 · Data Loading (train / val / test)

Original CUB train split → stratified 80/20 train/val.
**Validation** for all model selection. **Test** evaluated once at the end.


In [ ]:
# ============================
# Data Loading + Assertions
# ============================
def load_cub(data_dir):
    data_dir = Path(data_dir)
    images = {}
    with open(data_dir / 'images.txt') as f:
        for line in f:
            iid, path = line.strip().split(maxsplit=1); images[int(iid)] = path
    labels = {}
    with open(data_dir / 'image_class_labels.txt') as f:
        for line in f:
            iid, lbl = line.strip().split(); labels[int(iid)] = int(lbl) - 1
    is_train = {}
    with open(data_dir / 'train_test_split.txt') as f:
        for line in f:
            iid, s = line.strip().split(); is_train[int(iid)] = int(s) == 1
    attr_names = {}
    with open(data_dir / 'attributes' / 'attributes.txt') as f:
        for line in f:
            aid, name = line.strip().split(maxsplit=1); attr_names[int(aid)] = name

    # CUB rows are image_id, attribute_id, is_present, certainty, annotation_time.
    # We use all binary labels (ATTRIBUTE_LABEL_POLICY='all') and retain certainty
    # for auditing rather than silently discarding the annotation-quality metadata.
    rows = []
    with open(data_dir / 'attributes' / 'image_attribute_labels.txt') as f:
        for line in f:
            parts = line.split()
            assert len(parts) >= 4, f'Malformed attribute row: {line!r}'
            rows.append(tuple(map(int, parts[:4])))
    raw = np.asarray(rows, dtype=np.int32)
    n_img = len(images)
    assert raw.shape[0] == n_img * NUM_CONCEPTS, (
        f'Expected {n_img * NUM_CONCEPTS} rows, got {raw.shape[0]}')
    expected_iids = np.repeat(np.arange(1, n_img + 1), NUM_CONCEPTS)
    expected_aids = np.tile(np.arange(1, NUM_CONCEPTS + 1), n_img)
    assert np.array_equal(raw[:, 0], expected_iids), 'Attribute rows are not ordered by image ID.'
    assert np.array_equal(raw[:, 1], expected_aids), 'Attribute rows are not ordered by attribute ID.'
    attr_matrix = raw[:, 2].astype(np.float32).reshape(n_img, NUM_CONCEPTS)
    certainty_matrix = raw[:, 3].reshape(n_img, NUM_CONCEPTS)

    class_names = {}
    with open(data_dir / 'classes.txt') as f:
        for line in f:
            cid, name = line.strip().split(maxsplit=1); class_names[int(cid) - 1] = name
    return dict(images=images, labels=labels, is_train=is_train, attr_names=attr_names,
                attr_matrix=attr_matrix, certainty_matrix=certainty_matrix,
                class_names=class_names)


class CUB200(Dataset):
    def __init__(self, data_dir, meta, iids, transform=None):
        self.data_dir, self.meta, self.iids, self.transform = Path(data_dir), meta, iids, transform
    def __len__(self): return len(self.iids)
    def __getitem__(self, idx):
        iid = self.iids[idx]
        img = Image.open(self.data_dir / 'images' / self.meta['images'][iid]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.from_numpy(self.meta['attr_matrix'][iid - 1]), self.meta['labels'][iid]


_norm = dict(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
train_tf = transforms.Compose([transforms.Resize((256,256)), transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize(**_norm)])
eval_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(), transforms.Normalize(**_norm)])

meta = load_cub(DATA_DIR)
print(f'{len(meta["images"])} images, {len(meta["class_names"])} classes, {len(meta["attr_names"])} attributes')
print(f'Attribute-label policy: {ATTRIBUTE_LABEL_POLICY}; certainty distribution: '
      f'{dict(zip(*np.unique(meta["certainty_matrix"], return_counts=True)))}')

orig_train_iids = [iid for iid, t in meta['is_train'].items() if t]
test_iids = [iid for iid, t in meta['is_train'].items() if not t]
random.shuffle(orig_train_iids)
class_to_iids = defaultdict(list)
for iid in orig_train_iids: class_to_iids[meta['labels'][iid]].append(iid)
trn_iids, val_iids = [], []
for _, iids in class_to_iids.items():
    split = max(1, int(len(iids) * (1 - VAL_FRAC)))
    trn_iids.extend(iids[:split]); val_iids.extend(iids[split:])
print(f'Train: {len(trn_iids)}  Val: {len(val_iids)}  Test: {len(test_iids)}')

train_loader = DataLoader(CUB200(DATA_DIR, meta, trn_iids, train_tf), BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
train_eval_loader = DataLoader(CUB200(DATA_DIR, meta, trn_iids, eval_tf), BATCH_SIZE, shuffle=False,
                               num_workers=4, pin_memory=True)
val_loader = DataLoader(CUB200(DATA_DIR, meta, val_iids, eval_tf), BATCH_SIZE, shuffle=False,
                        num_workers=4, pin_memory=True)
test_loader = DataLoader(CUB200(DATA_DIR, meta, test_iids, eval_tf), BATCH_SIZE, shuffle=False,
                         num_workers=4, pin_memory=True)


## 2 · Axiom Generation (training data only)

Axiom literals selected from **training data only** via class-conditional attribute means.
Axiom structure is **locked** before any model training.


In [ ]:
# ============================
# Train-Only Axiom Generation
# ============================
def compute_train_literal_scores(meta, train_iids):
    class_sum = np.zeros((NUM_CLASSES, NUM_CONCEPTS), dtype=np.float64)
    class_cnt = np.zeros(NUM_CLASSES, dtype=np.int32)
    for iid in train_iids:
        lbl = meta['labels'][iid]
        class_sum[lbl] += meta['attr_matrix'][iid - 1]; class_cnt[lbl] += 1
    class_mean = class_sum / class_cnt[:, None]
    total_sum, total_cnt = class_sum.sum(0), class_cnt.sum()
    rest_mean = (total_sum[None, :] - class_sum) / (total_cnt - class_cnt)[:, None]
    signed_difference = class_mean - rest_mean
    # Sign determines literal polarity; magnitude ranks class-vs-rest discrimination.
    return class_mean, total_sum / total_cnt, signed_difference, np.abs(signed_difference)


def generate_axioms(signed_difference, scores, k):
    axioms = {}
    for c in range(NUM_CLASSES):
        top = np.argsort(scores[c])[-k:][::-1]
        axioms[c] = [(int(a), bool(signed_difference[c, a] >= 0.0)) for a in top]
    return axioms


class_attr_means, global_attr_mean, signed_literal_scores, literal_scores = compute_train_literal_scores(meta, trn_iids)
axiom_sets = {k: generate_axioms(signed_literal_scores, literal_scores, k)
              for k in [2, 3, 5, 8, 12]}
print(f'Example axioms K={TOP_K} for class 0 ({meta["class_names"][0]}):')
for aidx, pos in axiom_sets[TOP_K][0]:
    nm = meta['attr_names'].get(aidx + 1, f'a{aidx}')
    print(f'  {"+ " if pos else "- "}{nm}  '
          f'(class={class_attr_means[0,aidx]:.2f}, global={global_attr_mean[aidx]:.2f}, '
          f'class-minus-rest={signed_literal_scores[0,aidx]:+.2f})')

def build_attr_groups(attr_names):
    grps = defaultdict(list)
    for aid, name in attr_names.items():
        if '::' in name: grps[name.split('::')[0]].append(aid - 1)
    return [v for v in grps.values() if len(v) > 1]
ATTR_GROUPS = build_attr_groups(meta['attr_names'])
print(f'{len(ATTR_GROUPS)} attribute groups')


## 3 · Model Definitions

**v3 key fixes:**
- `SpeciesModel.forward(concepts, class_id)` — two-argument, per LTN binary predicate API
- `forward_logits()` returns unbounded values for `F.cross_entropy`
- `forward()` returns sigmoid for LTN truth values
- P_j restricted to K axiom concepts only — capacity-matched with pairwise tree


In [ ]:
# ============================
# Model Definitions
# ============================
class ConceptBottleneck(nn.Module):
    def __init__(self, n_concepts=312, pretrained=True):
        super().__init__()
        bb = models.resnet18(weights='DEFAULT' if pretrained else None)
        self.feat_dim = bb.fc.in_features; bb.fc = nn.Identity(); self.backbone = bb
        self.concept_head = nn.Sequential(nn.Linear(self.feat_dim, 512), nn.ReLU(), nn.Dropout(0.3),
                                          nn.Linear(512, n_concepts))
    def forward(self, x):
        logits = self.concept_head(self.backbone(x)); return logits, torch.sigmoid(logits)

class LinearCBM(nn.Module):
    def __init__(self, n_in=312, n_out=200): super().__init__(); self.fc = nn.Linear(n_in, n_out)
    def forward(self, c): return self.fc(c)

class MLPCBM(nn.Module):
    def __init__(self, n_in=312, n_out=200, h=512):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_in, h), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(h, h//2), nn.ReLU(), nn.Dropout(0.2), nn.Linear(h//2, n_out))
    def forward(self, c): return self.net(c)

def _axiom_buffers(axiom_map, n_classes):
    max_k = max(len(v) for v in axiom_map.values())
    idx = torch.zeros(n_classes, max_k, dtype=torch.long)
    pol = torch.ones(n_classes, max_k)
    for cid, cs in axiom_map.items():
        for j, (a, positive) in enumerate(cs): idx[cid, j], pol[cid, j] = a, 1. if positive else -1.
    return idx, pol

class PairwiseTreeCBM(nn.Module):
    """K-1 one-logit gates plus a class scale and bias: exactly K+1 raw parameters/class."""
    def __init__(self, axiom_map, n_concepts=312, n_classes=200):
        super().__init__()
        self.max_k = max(len(v) for v in axiom_map.values())
        idx, pol = _axiom_buffers(axiom_map, n_classes)
        self.register_buffer('idx', idx); self.register_buffer('pol', pol)
        self.gate_logit = nn.Parameter(torch.zeros(n_classes, self.max_k - 1))
        self.scale_raw = nn.Parameter(torch.full((n_classes,), 2.25))  # softplus ≈ 2.35
        self.bias = nn.Parameter(torch.zeros(n_classes))
    def forward(self, c):
        v = c[:, self.idx]
        v = torch.where(self.pol.unsqueeze(0) > 0, v, 1 - v)
        gi, cur = 0, v
        while cur.size(-1) > 1:
            k, n_pairs = cur.size(-1), cur.size(-1) // 2
            left, right = cur[:, :, 0::2][:, :, :n_pairs], cur[:, :, 1::2][:, :, :n_pairs]
            choose_and = torch.sigmoid(self.gate_logit[:, gi:gi+n_pairs]).unsqueeze(0)
            and_v, or_v = left * right, left + right - left * right
            combined = choose_and * and_v + (1 - choose_and) * or_v
            if k % 2: combined = torch.cat([combined, cur[:, :, -1:]], dim=-1)
            cur, gi = combined, gi + n_pairs
        return cur.squeeze(-1) * F.softplus(self.scale_raw).unsqueeze(0) + self.bias.unsqueeze(0)

class SpeciesModel(nn.Module):
    """Polarity-normalised K-literal linear predicate used by both CE and LTN paths."""
    def __init__(self, axiom_map, n_classes=200):
        super().__init__()
        max_k = max(len(v) for v in axiom_map.values())
        idx, pol = _axiom_buffers(axiom_map, n_classes)
        self.register_buffer('cidx', idx); self.register_buffer('cpol', pol)
        self.weight = nn.Parameter(torch.randn(n_classes, max_k) * 0.01)
        self.bias = nn.Parameter(torch.zeros(n_classes))
    def _class_ids(self, class_id, batch_size):
        cid = class_id.reshape(-1).long()
        if cid.numel() == 1: cid = cid.expand(batch_size)
        assert cid.numel() == batch_size, 'LTN class constant did not broadcast to the sample batch.'
        return cid
    def forward_logits(self, concepts, class_id):
        cid = self._class_ids(class_id, concepts.size(0))
        raw = concepts.gather(1, self.cidx[cid])
        selected = torch.where(self.cpol[cid] > 0, raw, 1 - raw)
        return (selected * self.weight[cid]).sum(-1, keepdim=True) + self.bias[cid].unsqueeze(-1)
    def forward(self, concepts, class_id): return torch.sigmoid(self.forward_logits(concepts, class_id))

def eval_all_class_logits(spec_model, concepts, n_classes=200):
    raw = concepts[:, spec_model.cidx]
    selected = torch.where(spec_model.cpol.unsqueeze(0) > 0, raw, 1 - raw)
    return (selected * spec_model.weight.unsqueeze(0)).sum(-1) + spec_model.bias.unsqueeze(0)

class ConceptExtract(nn.Module):
    def __init__(self, i): super().__init__(); self.i = i
    def forward(self, x): return x[:, self.i:self.i+1]


## 4 · Training

In [ ]:
# ============================
# Training Utilities
# ============================

def train_concept_model(model, trn_dl, val_dl, epochs, device):
    model.to(device)
    opt = optim.Adam([
        {'params': model.backbone.parameters(), 'lr': BACKBONE_LR},
        {'params': model.concept_head.parameters(), 'lr': CONCEPT_LR},
    ], weight_decay=WEIGHT_DECAY)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit  = nn.BCEWithLogitsLoss()
    best_loss, best_state = 1e9, None

    for ep in range(epochs):
        model.train(); tot=0; nb=0
        for imgs, attrs, _ in tqdm(trn_dl, leave=False, desc=f'Ep{ep+1}'):
            imgs, attrs = imgs.to(device), attrs.to(device)
            logits, _ = model(imgs)
            loss = crit(logits, attrs)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        sched.step()
        model.eval(); vl=0; vnb=0
        with torch.no_grad():
            for imgs, attrs, _ in val_dl:
                imgs, attrs = imgs.to(device), attrs.to(device)
                logits, _ = model(imgs)
                vl += crit(logits, attrs).item(); vnb += 1
        vl /= vnb
        if vl < best_loss:
            best_loss = vl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if (ep+1) % 5 == 0 or ep == 0:
            print(f'  Ep {ep+1}: trn={tot/nb:.4f} val={vl:.4f}')
    model.load_state_dict(best_state)
    return model


@torch.no_grad()
def extract_concepts(model, loader, device):
    """Extract concepts using a DETERMINISTIC loader (eval_tf, no shuffle)."""
    model.eval(); cs,ls,ats=[],[],[]
    for imgs, attrs, labels in tqdm(loader, leave=False, desc='Extract'):
        _, preds = model(imgs.to(device))
        cs.append(preds.cpu()); ls.append(labels); ats.append(attrs)
    return torch.cat(cs), torch.cat(ls), torch.cat(ats)


def eval_concept_quality(pred_c, true_a):
    """Mean per-attribute F1 and mean per-attribute AUROC."""
    p = (pred_c.numpy() > 0.5).astype(int)
    t = true_a.numpy().astype(int)
    pv = pred_c.numpy()
    # Per-attribute F1, then average
    f1s, aurocs = [], []
    for j in range(t.shape[1]):
        if t[:, j].std() == 0:
            continue  # skip constant attributes
        f1s.append(f1_score(t[:, j], p[:, j]))
        aurocs.append(roc_auc_score(t[:, j], pv[:, j]))
    return np.mean(f1s), np.mean(aurocs)


def train_ce_head(clf, trn_c, trn_l, val_c, val_l, epochs, device, name=''):
    clf.to(device)
    opt = optim.Adam(clf.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    trn_dl = DataLoader(TensorDataset(trn_c, trn_l), BATCH_SIZE, shuffle=True)
    val_dl = DataLoader(TensorDataset(val_c, val_l), BATCH_SIZE)
    best_acc, best_st = 0, None

    for ep in range(epochs):
        clf.train()
        for c,l in trn_dl:
            c,l=c.to(device),l.to(device)
            loss=F.cross_entropy(clf(c),l)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        clf.eval(); cor=0; ttl=0
        with torch.no_grad():
            for c,l in val_dl:
                c,l=c.to(device),l.to(device)
                cor+=(clf(c).argmax(1)==l).sum().item(); ttl+=l.size(0)
        acc=cor/ttl
        if acc > best_acc:
            best_acc=acc; best_st={k:v.cpu().clone() for k,v in clf.state_dict().items()}
        if (ep+1)%10==0 or ep==0:
            print(f'  [{name:15s}] Ep{ep+1:3d} val={acc:.4f} best={best_acc:.4f}')
    clf.load_state_dict(best_st); return clf, best_acc


In [ ]:
# ============================
# Genuine LTN-CBM Training
# ============================

def train_ltn_cbm(axiom_map, trn_c, trn_l, val_c, val_l,
                  epochs, device, lam=0.5, name='LTN-CBM'):
    """
    Genuine LTN training loop:
      - ltn.Predicate wrapping SpeciesModel (two-arg forward)
      - ltn.Connective: AndProd, NotStandard, ImpliesReichenbach
      - ltn.Quantifier: Forall with pMeanError
      - SatAgg over all per-class axioms
      - Hybrid loss: CE(logits, y) + lam * (1 - SatAgg)
        CE uses forward_logits() (unbounded); LTN uses forward() (sigmoid)
    """
    And_c     = ltn.Connective(ltn.fuzzy_ops.AndProd())
    Not_c     = ltn.Connective(ltn.fuzzy_ops.NotStandard())
    Implies_c = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())
    Forall_q  = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier='f')
    SatAgg_fn = ltn.fuzzy_ops.SatAgg(agg_op=ltn.fuzzy_ops.AggregPMeanError(p=2))

    # Concept extractors (fixed, no learnable params)
    used = set()
    for cs in axiom_map.values():
        for i, _ in cs: used.add(i)
    c_preds = {i: ltn.Predicate(ConceptExtract(i).to(device)) for i in used}

    # Learnable species predicate (capacity-matched: K concepts only)
    spec_model = SpeciesModel(axiom_map, NUM_CLASSES).to(device)
    spec_pred  = ltn.Predicate(spec_model)

    opt   = optim.Adam(spec_model.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    trn_dl = DataLoader(TensorDataset(trn_c, trn_l), BATCH_SIZE, shuffle=True)
    val_dl = DataLoader(TensorDataset(val_c, val_l), BATCH_SIZE)

    best_acc, best_st = 0, None
    hist = {'val_acc':[], 'ce':[], 'sat':[]}

    for ep in range(epochs):
        spec_model.train()
        ep_ce, ep_sat, nb = 0, 0, 0

        for (c, l) in trn_dl:
            c, l = c.to(device), l.to(device)
            opt.zero_grad()

            # ---- CE loss (unbounded logits) ----
            logits = eval_all_class_logits(spec_model, c, NUM_CLASSES)
            ce = F.cross_entropy(logits, l)

            # ---- SatAgg loss (genuine LTN axioms) ----
            axiom_list = []
            for j in l.unique().tolist():
                j = int(j)
                pos_mask = (l == j); neg_mask = ~pos_mask
                cls = ltn.Constant(torch.tensor([float(j)], device=device))

                # Positive axiom: forall x+: (c1 AND ... AND cK) -> P_j(x)
                x_pos = ltn.Variable(f'xp{j}', c[pos_mask])
                lits = axiom_map[j]
                idx0, pol0 = lits[0]
                term = c_preds[idx0](x_pos)
                if not pol0: term = Not_c(term)
                conj = term
                for idx_i, pol_i in lits[1:]:
                    t = c_preds[idx_i](x_pos)
                    if not pol_i: t = Not_c(t)
                    conj = And_c(conj, t)
                impl = Implies_c(conj, spec_pred(x_pos, cls))
                axiom_list.append(Forall_q(x_pos, impl))

                # Negative axiom: forall x-: NOT P_j(x)
                if neg_mask.sum() > 0:
                    x_neg = ltn.Variable(f'xn{j}', c[neg_mask])
                    axiom_list.append(Forall_q(x_neg, Not_c(spec_pred(x_neg, cls))))

            if axiom_list:
                sat_val = SatAgg_fn(*axiom_list)
                sat_loss = 1.0 - sat_val
            else:
                sat_loss = torch.tensor(0.0, device=device)

            total = ce + lam * sat_loss
            total.backward(); opt.step()
            ep_ce += ce.item(); ep_sat += sat_loss.item(); nb += 1

        sched.step()

        # ---- Validation (uses logits, not sigmoid) ----
        spec_model.eval(); cor=0; ttl=0
        with torch.no_grad():
            for c, l in val_dl:
                c, l = c.to(device), l.to(device)
                logits = eval_all_class_logits(spec_model, c, NUM_CLASSES)
                cor += (logits.argmax(1)==l).sum().item(); ttl += l.size(0)
        acc = cor / ttl
        hist['val_acc'].append(acc); hist['ce'].append(ep_ce/nb)
        hist['sat'].append(ep_sat/nb)
        if acc > best_acc:
            best_acc = acc
            best_st = {k:v.cpu().clone() for k,v in spec_model.state_dict().items()}
        if (ep+1)%5==0 or ep==0:
            print(f'  [{name:12s}] Ep{ep+1:3d}: ce={ep_ce/nb:.4f} '
                  f'sat={ep_sat/nb:.4f} val={acc:.4f} best={best_acc:.4f}')

    if best_st: spec_model.load_state_dict(best_st)
    return spec_model, hist, best_acc


In [ ]:
# ============================
# Phase 1: Concept Predictor
# ============================
print('=' * 60)
print('PHASE 1 - Backbone + Concept Predictor')
print('=' * 60)

concept_model = ConceptBottleneck(NUM_CONCEPTS)
concept_model = train_concept_model(
    concept_model, train_loader, val_loader, EPOCHS_CONCEPT, DEVICE)
torch.save(concept_model.state_dict(), OUTPUT_DIR / 'concept_model.pth')

# Extract using DETERMINISTIC loader (eval_tf, no augmentation)
print('Extracting concepts (deterministic) ...')
trn_c, trn_l, trn_a = extract_concepts(concept_model, train_eval_loader, DEVICE)
val_c, val_l, val_a = extract_concepts(concept_model, val_loader, DEVICE)
tst_c, tst_l, tst_a = extract_concepts(concept_model, test_loader, DEVICE)
print(f'Train: {trn_c.shape}  Val: {val_c.shape}  Test: {tst_c.shape}')

# Concept quality on VALIDATION only (no test inspection before final eval)
mf1_val, auroc_val = eval_concept_quality(val_c, val_a)
print(f'Concept quality (val): mean-per-attr-F1={mf1_val:.4f}  mean-AUROC={auroc_val:.4f}')

del concept_model; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# ============================
# Phase 2: Classifier Heads (validation-only model selection)
# ============================
print('=' * 60); print('PHASE 2 - CLASSIFIER HEADS'); print('=' * 60)
results = {}

for name, model, c_train, c_val in [
    ('Linear CBM', LinearCBM(), trn_c, val_c),
    ('MLP CBM', MLPCBM(), trn_c, val_c),
    ('Oracle CBM', LinearCBM(), trn_a, val_a),
]:
    print(f'\n--- {name} ---')
    model, best = train_ce_head(model, c_train, trn_l, c_val, val_l, EPOCHS_HEAD, DEVICE, name)
    results[name] = {'model': model, 'val': best, 'uses_gt': name == 'Oracle CBM'}

print(f'\n--- K-literal CE control K={TOP_K} ---')
ce_m, ce_h, ce_b = train_ltn_cbm(axiom_sets[TOP_K], trn_c, trn_l, val_c, val_l,
                                  EPOCHS_HEAD, DEVICE, lam=0.0, name=f'CE K={TOP_K}')
results[f'K-Literal CE K={TOP_K}'] = {'model': ce_m, 'val': ce_b, 'hist': ce_h, 'is_ltn': True}

print(f'\n--- Pairwise-Tree K={TOP_K} ---')
pt_m, pt_b = train_ce_head(PairwiseTreeCBM(axiom_sets[TOP_K], NUM_CONCEPTS, NUM_CLASSES),
                            trn_c, trn_l, val_c, val_l, EPOCHS_HEAD, DEVICE, f'PairTree K={TOP_K}')
results[f'PairTree K={TOP_K}'] = {'model': pt_m, 'val': pt_b}

print(f'\n--- LTN-CBM K={TOP_K} ---')
ltn_m, ltn_h, ltn_b = train_ltn_cbm(axiom_sets[TOP_K], trn_c, trn_l, val_c, val_l,
                                     EPOCHS_HEAD, DEVICE, LAMBDA_SAT, f'LTN K={TOP_K}')
results[f'LTN-CBM K={TOP_K}'] = {'model': ltn_m, 'val': ltn_b, 'hist': ltn_h, 'is_ltn': True}

print('\nVALIDATION RESULTS')
for n, r in results.items(): print(f'  {n:24s} val_acc = {r["val"]:.4f}')


In [ ]:
# ============================
# K Ablation Training (validation only — run before the sealed test cell)
# ============================
print('\n' + '=' * 60); print('ABLATION - K (VALIDATION ONLY)'); print('=' * 60)
k_val, k_models = {}, {}
for k in [2, 3, 5, 8, 12]:
    print(f'\n--- K = {k} ---')
    ce_m, _, ce_b = train_ltn_cbm(axiom_sets[k], trn_c, trn_l, val_c, val_l,
                                  EPOCHS_HEAD, DEVICE, lam=0.0, name=f'CE K={k}')
    ltn_m, _, ltn_b = train_ltn_cbm(axiom_sets[k], trn_c, trn_l, val_c, val_l,
                                    EPOCHS_HEAD, DEVICE, LAMBDA_SAT, name=f'LTN K={k}')
    pt_m, pt_b = train_ce_head(PairwiseTreeCBM(axiom_sets[k], NUM_CONCEPTS, NUM_CLASSES),
                               trn_c, trn_l, val_c, val_l, EPOCHS_HEAD, DEVICE, f'PT K={k}')
    k_val[k] = {'ce_val': ce_b, 'ltn_val': ltn_b, 'pt_val': pt_b}
    k_models[k] = {'ce': ce_m, 'ltn': ltn_m, 'pt': pt_m}


In [ ]:
# ============================
# K Ablation Training (validation only — run before the sealed test cell)
# ============================
print('\n' + '=' * 60); print('ABLATION - K (VALIDATION ONLY)'); print('=' * 60)
k_val, k_models = {}, {}
for k in [2, 3, 5, 8, 12]:
    print(f'\n--- K = {k} ---')
    ce_m, _, ce_b = train_ltn_cbm(axiom_sets[k], trn_c, trn_l, val_c, val_l,
                                  EPOCHS_HEAD, DEVICE, lam=0.0, name=f'CE K={k}')
    ltn_m, _, ltn_b = train_ltn_cbm(axiom_sets[k], trn_c, trn_l, val_c, val_l,
                                    EPOCHS_HEAD, DEVICE, LAMBDA_SAT, name=f'LTN K={k}')
    pt_m, pt_b = train_ce_head(PairwiseTreeCBM(axiom_sets[k], NUM_CONCEPTS, NUM_CLASSES),
                               trn_c, trn_l, val_c, val_l, EPOCHS_HEAD, DEVICE, f'PT K={k}')
    k_val[k] = {'ce_val': ce_b, 'ltn_val': ltn_b, 'pt_val': pt_b}
    k_models[k] = {'ce': ce_m, 'ltn': ltn_m, 'pt': pt_m}


# ============================
# SEALED TEST EVALUATION (run once, after every model above is fixed)
# ============================
print('=' * 60); print('FINAL TEST EVALUATION (run once)'); print('=' * 60)

def test_accuracy(model, concepts, labels, device, is_ltn=False):
    model.eval()
    with torch.no_grad():
        c = concepts.to(device)
        logits = eval_all_class_logits(model, c, NUM_CLASSES) if is_ltn else model.to(device)(c)
        return (logits.argmax(1) == labels.to(device)).float().mean().item()

def intervention_curve(model, concepts, true_attrs, labels, fracs, device, is_ltn=False, n_trials=5):
    model.eval(); n_c, res = concepts.size(1), {}
    for fi, frac in enumerate(fracs):
        accs = []
        for trial in range(n_trials):
            generator = torch.Generator().manual_seed(SEED + fi * n_trials + trial)
            cc, nf = concepts.clone(), int(n_c * frac)
            for i in range(len(cc)):
                idx = torch.randperm(n_c, generator=generator)[:nf]
                cc[i, idx] = true_attrs[i, idx]
            with torch.no_grad():
                c = cc.to(device)
                logits = eval_all_class_logits(model, c, NUM_CLASSES) if is_ltn else model.to(device)(c)
                accs.append((logits.argmax(1) == labels.to(device)).float().mean().item())
        res[frac] = float(np.mean(accs))
    return res

def antecedent_stats(axiom_map, concepts, labels, threshold=0.5):
    pos_means, neg_means, coverage = [], [], []
    for j in range(NUM_CLASSES):
        lits = axiom_map[j]
        def conjunction(c):
            vals = [c[:, idx] if pol else 1 - c[:, idx] for idx, pol in lits]
            return torch.stack(vals, -1).prod(-1)
        pos, neg = concepts[labels == j], concepts[labels != j]
        if len(pos) and len(neg):
            pos_c, neg_c = conjunction(pos), conjunction(neg)
            pos_means.append(pos_c.mean().item()); neg_means.append(neg_c.mean().item())
            coverage.append((pos_c > threshold).float().mean().item())
    return {'positive_mean': float(np.mean(pos_means)), 'negative_mean': float(np.mean(neg_means)),
            'coverage': float(np.mean(coverage))}

fracs, test_results, interv_results = [0.0, 0.1, 0.25, 0.5, 0.75, 1.0], {}, {}
for name, r in results.items():
    is_ltn, uses_gt, model = r.get('is_ltn', False), r.get('uses_gt', False), r['model']
    c_in = tst_a if uses_gt else tst_c
    test_results[name] = test_accuracy(model, c_in, tst_l, DEVICE, is_ltn)
    if not uses_gt:  # Oracle is already fully intervened; do not plot a meaningless flat curve.
        interv_results[name] = intervention_curve(model, c_in, tst_a, tst_l, fracs, DEVICE, is_ltn)
print('\nTest accuracies:')
for n, a in test_results.items(): print(f'  {n:24s} test_acc = {a:.4f}')
print('Oracle CBM is excluded from intervention curves because it already receives GT attributes.')

mf1_tst, auroc_tst = eval_concept_quality(tst_c, tst_a)
print(f'\nConcept quality (test): mean-per-attr-F1={mf1_tst:.4f}  mean-AUROC={auroc_tst:.4f}')

# All ablation models were fixed before this cell. Evaluate each exactly once.
k_test = {}
for k in sorted(k_models):
    stats = antecedent_stats(axiom_sets[k], tst_c, tst_l)
    k_test[k] = {**k_val[k],
        'ce_test': test_accuracy(k_models[k]['ce'], tst_c, tst_l, DEVICE, True),
        'ltn_test': test_accuracy(k_models[k]['ltn'], tst_c, tst_l, DEVICE, True),
        'pt_test': test_accuracy(k_models[k]['pt'], tst_c, tst_l, DEVICE),
        'depth': math.ceil(math.log2(max(k, 2))), **stats}
print('\nK ablation test results:')
for k, r in sorted(k_test.items()):
    print(f'  K={k:2d} CE={r["ce_test"]:.4f} LTN={r["ltn_test"]:.4f} '
          f'PT={r["pt_test"]:.4f} cov={r["coverage"]:.3f}')


## 6 · Visualisation

Eight publication-quality figures covering all experimental dimensions:
1. **Model accuracy comparison** (bar chart with value labels)
2. **Random concept intervention curves** (line plot per model)
3. **K ablation: accuracy + antecedent coverage** (dual-axis)
4. **LTN training decomposition** (CE vs SatAgg loss per epoch)
5. **LTN vs CE-only accuracy delta per K** (the primary causal comparison)
6. **Antecedent selectivity analysis** (pos/neg conjunction means)
7. **Concept predictor quality per attribute group** (grouped bar)
8. **Sample axiom visualisation** (top-10 classes, K literals as heatmap)


In [ ]:
# ============================
# Figure 1-4: Core Results (2x2)
# ============================
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'figure.dpi': 120,
})

fig, axes = plt.subplots(2, 2, figsize=(18, 13))
fig.suptitle('LTN-CBM v3 \u2014 CUB-200-2011 (Controlled Training-Signal Study)',
             fontsize=16, fontweight='bold', y=1.01)

# ---- 1: Test accuracy bar chart ----
ax = axes[0, 0]
names = list(test_results.keys())
accs  = [test_results[n] for n in names]
palette = plt.cm.viridis(np.linspace(0.2, 0.85, len(names)))
bars = ax.bar(range(len(names)), accs, color=palette, edgecolor='white', lw=0.8)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Test Accuracy')
ax.set_title('1. Classifier Head Comparison')
ax.set_ylim(0, max(accs) * 1.15)
for b, a in zip(bars, accs):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+.005,
            f'{a:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)

# ---- 2: Intervention curves ----
ax = axes[0, 1]
markers = ['o','s','D','^','v','<','>','p']
for i, (n, ir) in enumerate(interv_results.items()):
    fs = sorted(ir.keys())
    mk = markers[i % len(markers)]
    ax.plot([f*100 for f in fs], [ir[f] for f in fs],
            f'{mk}-', label=n, lw=2, ms=6, alpha=0.85)
ax.set_xlabel('% Concepts Replaced with GT')
ax.set_ylabel('Test Accuracy')
ax.set_title('2. Random Concept Intervention Curve')
ax.legend(fontsize=7, loc='lower right', framealpha=0.9)
ax.spines[['top','right']].set_visible(False)
ax.grid(alpha=0.3)

# ---- 3: K ablation + coverage ----
ax = axes[1, 0]
ks = sorted(k_test.keys())
ax.plot(ks, [k_test[k]['ltn_test'] for k in ks], 's-',
        lw=2.5, ms=8, color='#2196F3', label='LTN-CBM (CE+SatAgg)', zorder=3)
ax.plot(ks, [k_test[k]['pt_test'] for k in ks], 'D-',
        lw=2.5, ms=8, color='#FF5722', label='PairTree (CE)', zorder=3)
ax.set_xlabel('K (concepts per axiom)')
ax.set_ylabel('Test Accuracy')
ax.set_title('3. K Ablation: Accuracy & Coverage')
ax.set_xticks(ks)
ax.legend(loc='upper left', fontsize=8)
ax.spines[['top']].set_visible(False)
ax.grid(alpha=0.3)
ax2 = ax.twinx()
ax2.fill_between(ks, [k_test[k]['coverage'] for k in ks],
                 alpha=0.15, color='gray', label='Antecedent Coverage')
ax2.plot(ks, [k_test[k]['coverage'] for k in ks],
         'x--', color='gray', lw=1.5, ms=7)
ax2.set_ylabel('Coverage (fraction > 0.5)', color='gray')
ax2.set_ylim(0, 1.05)
ax2.legend(loc='lower right', fontsize=7)
ax2.spines[['top']].set_visible(False)

# ---- 4: LTN training decomposition ----
ax = axes[1, 1]
ltn_key = f'LTN-CBM K={TOP_K}'
if ltn_key in results and 'hist' in results[ltn_key]:
    h = results[ltn_key]['hist']
    eps = list(range(1, len(h['ce'])+1))
    ax.plot(eps, h['ce'], lw=2.5, color='#E91E63', label='CE loss')
    ax.plot(eps, h['sat'], lw=2.5, color='#9C27B0', label='1 \u2212 SatAgg')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(f'4. LTN-CBM K={TOP_K} Training Decomposition')
    ax.legend(fontsize=9)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(alpha=0.3)
    ax3 = ax.twinx()
    ax3.plot(eps, h['val_acc'], '--', lw=1.5, color='#4CAF50', alpha=0.7,
             label='Val Accuracy')
    ax3.set_ylabel('Val Accuracy', color='#4CAF50')
    ax3.legend(loc='center right', fontsize=7)
    ax3.spines[['top']].set_visible(False)
else:
    ax.text(0.5, 0.5, 'No LTN training history', transform=ax.transAxes,
            ha='center', va='center', fontsize=12, color='gray')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_core_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {OUTPUT_DIR / "fig_core_results.png"}')


In [ ]:
# ============================
# Figure 5-6: Causal Comparison & Antecedent Analysis
# ============================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Primary Causal Comparison: CE-only vs CE+SatAgg (same SpeciesModel)',
             fontsize=14, fontweight='bold', y=1.02)

ax = axes[0]
ks = sorted(k_test.keys())
deltas = [k_test[k]['ltn_test'] - k_test[k]['ce_test'] for k in ks]
colours = ['#4CAF50' if d >= 0 else '#F44336' for d in deltas]
bars = ax.bar(ks, deltas, color=colours, edgecolor='white', width=0.6, alpha=0.85)
ax.axhline(0, color='black', lw=1)
for b, d in zip(bars, deltas):
    ax.text(b.get_x()+b.get_width()/2, b.get_height() + (0.001 if d >= 0 else -0.003),
            f'{d:+.3f}', ha='center', va='bottom' if d >= 0 else 'top', fontsize=10, fontweight='bold')
ax.set(xlabel='K (concepts per axiom)', ylabel='Accuracy Delta (LTN − CE-only)',
       title='5. SatAgg Regularisation Effect')
ax.set_xticks(ks); ax.spines[['top','right']].set_visible(False); ax.grid(axis='y', alpha=0.3)

ax = axes[1]
pos = [k_test[k]['positive_mean'] for k in ks]
neg = [k_test[k]['negative_mean'] for k in ks]
covs = [k_test[k]['coverage'] for k in ks]
x, w = np.arange(len(ks)), 0.25
ax.bar(x-w, pos, w, label='Positive class mean', color='#2196F3', alpha=0.8)
ax.bar(x, neg, w, label='Negative class mean', color='#FF5722', alpha=0.8)
ax.bar(x+w, covs, w, label='Positive coverage (>0.5)', color='#4CAF50', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels([str(k) for k in ks])
ax.set(xlabel='K', ylabel='Product-conjunction value', title='6. Antecedent Selectivity')
ax.legend(fontsize=8); ax.spines[['top','right']].set_visible(False); ax.grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'fig_causal_comparison.png', dpi=150, bbox_inches='tight')
plt.show(); print(f'Saved -> {OUTPUT_DIR / "fig_causal_comparison.png"}')


In [ ]:
# ============================
# Figure 7: Concept Quality per Attribute Group
# ============================
fig, ax = plt.subplots(figsize=(14, 5))

group_names, group_f1s = [], []
grps_raw = defaultdict(list)
for aid, name in meta['attr_names'].items():
    if '::' in name:
        grps_raw[name.split('::')[0]].append(aid - 1)

val_pred = val_c.numpy()
val_true = val_a.numpy().astype(int)

for gname, indices in sorted(grps_raw.items()):
    f1s = []
    for j in indices:
        if val_true[:, j].std() == 0:
            continue
        f1s.append(f1_score(val_true[:, j], (val_pred[:, j] > 0.5).astype(int)))
    if f1s:
        group_names.append(gname.replace('has_', '').replace('_', ' ').title())
        group_f1s.append(np.mean(f1s))

x = np.arange(len(group_names))
colours = plt.cm.coolwarm(np.array(group_f1s))
bars = ax.bar(x, group_f1s, color=colours, edgecolor='white', lw=0.5)
ax.axhline(np.mean(group_f1s), color='red', ls='--', lw=1.5, alpha=0.7,
           label=f'Mean = {np.mean(group_f1s):.3f}')
ax.set_xticks(x)
ax.set_xticklabels(group_names, rotation=55, ha='right', fontsize=7)
ax.set_ylabel('Mean Per-Attribute F1 (val)')
ax.set_title('7. Concept Predictor Quality by Attribute Group', fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_concept_quality.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {OUTPUT_DIR / "fig_concept_quality.png"}')


In [ ]:
# ============================
# Figure 8: Sample Axiom Visualization (heatmap)
# ============================
n_show = min(10, NUM_CLASSES)
k_show = TOP_K
ax_map = axiom_sets[k_show]

fig, ax = plt.subplots(figsize=(12, max(4, n_show * 0.5)))

class_labels = []
mat = np.zeros((n_show, k_show))

for c in range(n_show):
    lits = ax_map[c]
    cname = meta['class_names'].get(c, f'Class {c}')
    cname = cname.split('.')[-1] if '.' in cname else cname
    class_labels.append(cname[:25])
    for j, (aidx, pol) in enumerate(lits):
        mat[c, j] = class_attr_means[c, aidx] if pol else (1 - class_attr_means[c, aidx])

im = ax.imshow(mat, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_yticks(range(n_show))
ax.set_yticklabels(class_labels, fontsize=8)
ax.set_xticks(range(k_show))
ax.set_xticklabels([f'Lit {i+1}' for i in range(k_show)], fontsize=9)
ax.set_title(f'8. Axiom Literal Strengths (K={k_show}, top {n_show} classes)',
             fontweight='bold')

for c in range(n_show):
    for j, (aidx, pol) in enumerate(ax_map[c]):
        aname = meta['attr_names'].get(aidx+1, f'a{aidx}')
        short = aname.split('::')[-1] if '::' in aname else aname
        short = short[:12]
        sign = '+' if pol else '-'
        ax.text(j, c, f'{sign}{short}', ha='center', va='center',
                fontsize=6, color='black',
                bbox=dict(boxstyle='round,pad=0.1', facecolor='white', alpha=0.6))

plt.colorbar(im, ax=ax, label='Class-conditional mean (polarity-adjusted)', shrink=0.8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_axiom_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved -> {OUTPUT_DIR / "fig_axiom_heatmap.png"}')


In [ ]:
# ============================
# Save all results to JSON
# ============================
all_out = {
    'test': test_results,
    'intervention': {n: {str(k):v for k,v in ir.items()} for n,ir in interv_results.items()},
    'k_ablation': {str(k): {kk:vv for kk,vv in r.items()}
                   for k,r in k_test.items()},
    'concept_quality_val': {'mean_per_attr_f1': mf1_val, 'mean_auroc': auroc_val},
    'concept_quality_test': {'mean_per_attr_f1': mf1_tst, 'mean_auroc': auroc_tst},
    'antecedent_coverage': {str(k): k_test[k]['coverage'] for k in k_test},
}
with open(OUTPUT_DIR / 'results_v3.json', 'w') as f:
    json.dump(all_out, f, indent=2)
print(f'Saved results -> {OUTPUT_DIR / "results_v3.json"}')


## 7 · Summary & Limitations

### What This Notebook Implements
- A genuine **LTNtorch** `Predicate`/`Connective`/`Quantifier`/`SatAgg` training path.
- A primary **same-head CE-only control** (`λ=0`) with identical polarity-normalised K inputs.
- Pairwise Tree with exactly K+1 raw parameters per class, matching `SpeciesModel`.
- Train-only, signed class-vs-rest literal selection; negative literals are normalised before both heads.
- Full attribute-row ordering validation and an explicit all-label certainty policy.
- Every K-ablation model is trained on validation before one final, sealed test-evaluation cell.
- Deterministic concept extraction and seeded random-intervention draws. Oracle is excluded from
  intervention curves because it already receives ground-truth attributes.

### Honest Limitations
1. A sufficient implication is a regulariser, not a proof that the displayed antecedent is the
   complete decision rule; report predicate weights alongside each axiom.
2. Product conjunction can have low coverage and weak gradients at larger K. Coverage and
   positive/negative antecedent means are reported to make that failure mode visible.
3. CUB-200 is propositional: ∀x ranges over samples, not objects in a relational scene.
4. Pairwise-Tree is this controlled baseline, not the published LogicCBM CP/G-matrix.
5. Random intervention is not LogicCBMs' CCG metric.
